# 06.2 — VAE and the ELBO

The encoder isn't just a compressor — it's a **variational autoencoder**. Instead of mapping a trial to a fixed latent vector, it maps it to a *distribution* (a mean and variance), samples from it, and the decoder reconstructs from the sample. This buys a smooth, well-structured latent space, at the cost of two coupled losses: **reconstruction** (rebuild the input) and **KL** (keep the latent close to a unit Gaussian). Together they are the **ELBO** — the VAE's training objective. This notebook is the foundation the rest of Module 06 builds on.

This notebook covers:

* The VAE idea: encode to a distribution, sample, decode.
* The reparameterization trick — how to backprop through a random sample.
* The two ELBO halves: reconstruction (MSE/MAE) and KL divergence.
* The production `SamplingLayer`, `masked_mse_reconstruction_loss`, `kl_divergence_loss`.

**Prerequisites:** [04.5 the bottleneck](../04_architecture/04.5_the_bottleneck.ipynb) (the 2×latent width), [02.5 autograd](../02_numpy_and_pytorch_basics/02.5_autograd_basics.ipynb), [06.1 overview](06.1_multi_task_losses_overview.ipynb).

## Section 1 — What MATLAB does

`cgg_lossELBO_v2` computes the two-part objective, and `cgg_samplingLayer` draws the latent:

```matlab
% Sampling: split the bottleneck output into mean + log-variance, then sample
mu         = stats(1:latent, :);
logSigmaSq = stats(latent+1:end, :);
Z = mu + exp(0.5 * logSigmaSq) .* randn(size(mu));      % reparameterization

% ELBO = reconstruction + KL
Loss_Rec = 0.5 * l2loss(Yrec, T, 'Mask', ~isnan(T));    % masked MSE
Loss_KL  = -0.5 * sum(1 + logSigmaSq - mu.^2 - exp(logSigmaSq));
```

The `randn` inside the sampling is the whole subtlety — you can't backprop through a raw random draw, which is why the reparameterization form (`mu + sigma·eps`) exists. The port matches this exactly.

## Section 2 — The Python concepts you need

### 2.1 — Encode to a distribution, not a point

The bottleneck ([04.5 §2.3](../04_architecture/04.5_the_bottleneck.ipynb)) outputs `2 × latent_dim` numbers — the first half is the mean μ, the second the log-variance log σ². The latent is then a *sample* from `N(μ, σ²)`, not μ itself. The `SamplingLayer` does the split and the draw:

In [ ]:
import torch
from neural_data_decoding.models.layers.sampling import SamplingLayer

sampler = SamplingLayer()
latent_dim = 3

# Bottleneck output: (batch, time, 2×latent) = [mu | logvar] concatenated
stats = torch.randn(4, 5, 2 * latent_dim)
sampler.train()                                    # training mode → stochastic
z, mu, logvar = sampler(stats)
print("bottleneck stats:", tuple(stats.shape), "= [mu | logvar]")
print("z (sampled latent):", tuple(z.shape))
print("mu, logvar:        ", tuple(mu.shape), tuple(logvar.shape))

The layer splits the channel axis in half — mean and log-variance — and returns all three (`z`, `mu`, `logvar`) because the ELBO needs μ and logσ² for the KL term, not just the sample. Log-variance (not variance) is parameterized because it's unconstrained (variance must be positive; its log can be any real), which keeps the network's output unrestricted.

### 2.2 — The reparameterization trick

Here's the problem: `z = sample from N(μ, σ²)` involves randomness, and you can't compute a gradient of a random draw. The trick: rewrite it as `z = μ + σ·ε` where `ε ~ N(0,1)` is drawn *outside* the graph. Now `z` is a deterministic (differentiable!) function of μ and σ, with ε just a random constant — gradients flow to μ and σ, the randomness doesn't block them:

In [ ]:
# The reparameterization, by hand — and proof gradients flow to mu and logvar:
mu = torch.zeros(3, requires_grad=True)
logvar = torch.zeros(3, requires_grad=True)
eps = torch.randn(3)                               # drawn OUTSIDE the graph (a constant)

z = mu + eps * torch.exp(0.5 * logvar)             # differentiable in mu, logvar
z.sum().backward()
print("gradient flows to mu?    ", mu.grad is not None, mu.grad.tolist())
print("gradient flows to logvar?", logvar.grad is not None)
print("→ randomness (eps) is a constant; mu and logvar are trainable. This is the trick.")

Without it, the sampling would be a gradient dead-end and the encoder couldn't learn. The `SamplingLayer.forward` you saw in §2.1 does exactly `z = mu + eps * exp(0.5*logvar)` in training mode — line for line the MATLAB `Z = mu + exp(0.5*logSigmaSq).*randn(...)`.

### 2.3 — The reconstruction half

In [ ]:
from neural_data_decoding.training.losses.elbo import masked_mse_reconstruction_loss

# Decoder output vs the original target. Reconstruction loss = masked MSE.
target = torch.randn(4, 10)
perfect = target.clone()
noisy = target + 0.5 * torch.randn(4, 10)

print("perfect reconstruction loss:", masked_mse_reconstruction_loss(perfect, target).item())
print("noisy   reconstruction loss:", round(masked_mse_reconstruction_loss(noisy, target).item(), 3))

Reconstruction is a masked MSE ([06.10](06.10_nan_masked_reconstruction.ipynb) covers the NaN-mask in depth) — zero when the decoder rebuilds the input perfectly, growing with error. It's normalized by *batch size* (not by the number of valid elements) — a MATLAB-parity detail verified empirically against the reference (the plan's original note was wrong; the probe caught it, [06.10](06.10_nan_masked_reconstruction.ipynb)).

### 2.4 — The KL half

In [ ]:
from neural_data_decoding.training.losses.elbo import kl_divergence_loss

# KL measures how far N(mu, sigma²) is from N(0, 1). Zero when they match.
mu_match = torch.zeros(4, 3); logvar_match = torch.zeros(4, 3)      # exactly N(0,1)
mu_off = torch.ones(4, 3) * 2; logvar_off = torch.ones(4, 3)       # shifted + wider

print("KL when latent = N(0,1) exactly:", kl_divergence_loss(mu_match, logvar_match).item())
print("KL when latent is off:          ", round(kl_divergence_loss(mu_off, logvar_off).item(), 3))

KL divergence measures how far the encoder's latent distribution is from a unit Gaussian — zero when μ=0 and σ=1 (`logvar=0`), positive otherwise. It's a *regularizer*: it pulls the latent toward a smooth, standard shape, preventing the encoder from spreading codes arbitrarily. The formula `-0.5·Σ(1 + logσ² − μ² − σ²)` is the closed-form KL between two Gaussians, summed over the latent channels then averaged over batch×time — matching `cgg_lossELBO_v2`.

### 2.5 — ELBO = reconstruction + KL

In [ ]:
# The VAE objective is the two halves together (each will be weighted + normalized, 06.1 §2.2):
decoder_out = torch.randn(4, 3)
recon = masked_mse_reconstruction_loss(decoder_out, torch.randn(4, 3))
kl = kl_divergence_loss(torch.randn(4, 3), torch.zeros(4, 3))
elbo = recon + kl
print(f"reconstruction {recon.item():.3f} + KL {kl.item():.3f} = ELBO {elbo.item():.3f}")
print("Reconstruction wants accuracy; KL wants a tidy latent. Their balance IS the VAE.")

The tension is the point: reconstruction alone would memorize; KL alone would ignore the data. Balanced (via the weights + normalization of [06.1](06.1_multi_task_losses_overview.ipynb)/[06.4](06.4_the_ema_prior_normalization_deep_dive.ipynb)), they yield a latent that's both informative and well-structured. KL annealing ([Module 07](../README.md)) ramps the KL weight up over training so the model learns to reconstruct first, then tidies the latent.

## Section 3 — The `neural_data_decoding` implementation

`SamplingLayer.forward` — the reparameterization in production, with the train/eval branch:

In [ ]:
from pathlib import Path
src = Path("../../src/neural_data_decoding/models/layers/sampling.py").read_text().splitlines()
i = next(j for j, line in enumerate(src) if "if self.training:" in line)
for k in range(i - 4, min(i + 7, len(src))):
    print(f"{k + 1:4} | {src[k]}")

Things to spot:

* `z = mu + eps * torch.exp(0.5 * logvar)` in training — §2.2's reparameterization, exactly.
* The `else: z = mu` branch — at **inference** the layer returns the mean deterministically (no sampling), Critical Note #35, deep-dived in [06.13](06.13_sampling_layer_deterministic_at_inference.ipynb). Same input → same output at eval time, unlike textbook VAEs.
* `mu.narrow(...)` splits the channel axis — the mean is the first half, log-variance the second (§2.1).
* The layer is stateless (no learnable parameters) — pure computation, so it needs no weights in the checkpoint ([05.7 §3](../05_training_loop/05.7_batch_norm_state_synchronization.ipynb)).

## Section 4 — Hands-on exercises

### Exercise 4.1 — KL is zero exactly at N(0,1)

Confirm the KL loss is (numerically) zero when μ=0 and logvar=0, and strictly positive for any deviation in either.

In [ ]:
# Reveal:
z = torch.zeros(2, 4)
print("KL(mu=0, logvar=0):        ", kl_divergence_loss(z, z).item())
print("KL(mu=0.1, logvar=0):      ", round(kl_divergence_loss(z + 0.1, z).item(), 4))
print("KL(mu=0, logvar=0.1):      ", round(kl_divergence_loss(z, z + 0.1).item(), 4))
print("→ any deviation from N(0,1) costs KL.")

### Exercise 4.2 — sampling is stochastic in train, deterministic in eval

Show that two forward passes of the SamplingLayer give DIFFERENT z in train mode but IDENTICAL z in eval mode (the Note #35 behavior).

In [ ]:
# Reveal:
stats = torch.randn(2, 4)          # [mu | logvar], latent=2
s = SamplingLayer()

s.train()
z1, _, _ = s(stats); z2, _, _ = s(stats)
print("train mode — two draws equal?", torch.equal(z1, z2))     # False — stochastic

s.eval()
z3, _, _ = s(stats); z4, _, _ = s(stats)
print("eval  mode — two draws equal?", torch.equal(z3, z4))     # True — z = mu

## Section 5 — Common errors

### Gradients don't reach the encoder through sampling

You sampled with `torch.normal(mu, sigma)` (a raw draw, no gradient path) instead of the reparameterization `mu + sigma*eps` (§2.2). Only the reparameterized form is differentiable.

### KL loss is negative

The formula sign is flipped, or you passed variance where log-variance is expected. KL is always ≥ 0; a negative value means a bug in the term.

### Reconstruction loss ignores NaN incorrectly

The masked loss derives its mask from the *original* NaN-bearing target ([02.8 §2.4](../02_numpy_and_pytorch_basics/02.8_nan_handling.ipynb), [06.10](06.10_nan_masked_reconstruction.ipynb)). Feed it the raw target, not a zero-filled one, or the mask is wrong.

### Latent collapses (KL → 0, model ignores the data)

"Posterior collapse" — KL weight too high too early, so the encoder finds it cheapest to output N(0,1) and ignore the input. KL annealing ([Module 07](../README.md)) ramps the KL weight up gradually to avoid this.

### Eval outputs vary run to run

The SamplingLayer wasn't put in `eval()` mode, so it's still sampling. `model.eval()` switches it to the deterministic `z = mu` ([05.7](../05_training_loop/05.7_batch_norm_state_synchronization.ipynb), [06.13](06.13_sampling_layer_deterministic_at_inference.ipynb)).

## Section 6 — Further reading

- [Auto-Encoding Variational Bayes (Kingma & Welling)](https://arxiv.org/abs/1312.6114) — the VAE + reparameterization paper.
- [A tutorial on VAEs (Doersch)](https://arxiv.org/abs/1606.05908) — the ELBO derived gently.
- [`src/neural_data_decoding/training/losses/elbo.py`](../../src/neural_data_decoding/training/losses/elbo.py) — the two loss halves.
- [`src/neural_data_decoding/models/layers/sampling.py`](../../src/neural_data_decoding/models/layers/sampling.py) — the sampling layer.

Next up: **[06.3 stochastic vs deterministic placement](06.3_stochastic_vs_deterministic_placement.ipynb)** — the two graph topologies and why Optimal uses Stochastic.